In [1]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("PySparkGuide")
    .master("spark://bd-spark-master:7077")
    .getOrCreate()
)

data = [
    (1, "Alice", "HR", 5000, "2023-01-15", None),
    (2, "Bob", "IT", 6000, "2023-02-20", "bob@x.com"),
    (3, "Carol", "IT", 7000, "2023-03-10", "carol@x.com"),
    (4, "Dave", "HR", None, "2023-04-05", None),
    (5, "Eve", "IT", 6000, "2023-05-01", "eve@x.com"),
]
columns = ["id", "name", "dept", "salary", "join_date", "email"]
df = spark.createDataFrame(data, columns)
df.show()
spark
print("Application name:", spark.sparkContext.appName)
print("Application ID:", spark.sparkContext.applicationId)
print("Spark UI:", spark.sparkContext.uiWebUrl)

+---+-----+----+------+----------+-----------+
| id| name|dept|salary| join_date|      email|
+---+-----+----+------+----------+-----------+
|  1|Alice|  HR|  5000|2023-01-15|       null|
|  2|  Bob|  IT|  6000|2023-02-20|  bob@x.com|
|  3|Carol|  IT|  7000|2023-03-10|carol@x.com|
|  4| Dave|  HR|  null|2023-04-05|       null|
|  5|  Eve|  IT|  6000|2023-05-01|  eve@x.com|
+---+-----+----+------+----------+-----------+

Application name: PySparkGuide
Application ID: app-20260905174722-0000
Spark UI: http://host.docker.internal:4040


In [2]:
# Basic aggregations
df.groupBy("dept").agg(
    F.count("*").alias("emp_count"),
    F.sum("salary").alias("total_salary"),
    F.avg("salary").alias("avg_salary"),
    F.max("salary").alias("max_salary"),
    F.min("salary").alias("min_salary")
).show()

# Multiple group keys
df.groupBy("dept", "name").count().show()

# Pivot: dept becomes rows, one column per distinct value, values summed
df.groupBy("dept").pivot("name").sum("salary").show()

# Pivot with explicit values (faster — avoids scanning to find distinct values)
df.groupBy("dept").pivot("name", ["Alice", "Bob", "Carol"]).sum("salary").show()

+----+---------+------------+-----------------+----------+----------+
|dept|emp_count|total_salary|       avg_salary|max_salary|min_salary|
+----+---------+------------+-----------------+----------+----------+
|  HR|        2|        5000|           5000.0|      5000|      5000|
|  IT|        3|       19000|6333.333333333333|      7000|      6000|
+----+---------+------------+-----------------+----------+----------+

+----+-----+-----+
|dept| name|count|
+----+-----+-----+
|  IT|  Eve|    1|
|  IT|Carol|    1|
|  HR| Dave|    1|
|  IT|  Bob|    1|
|  HR|Alice|    1|
+----+-----+-----+

+----+-----+----+-----+----+----+
|dept|Alice| Bob|Carol|Dave| Eve|
+----+-----+----+-----+----+----+
|  HR| 5000|null| null|null|null|
|  IT| null|6000| 7000|null|6000|
+----+-----+----+-----+----+----+

+----+-----+----+-----+
|dept|Alice| Bob|Carol|
+----+-----+----+-----+
|  HR| 5000|null| null|
|  IT| null|6000| 7000|
+----+-----+----+-----+



In [3]:
dept_info = spark.createDataFrame(
    [("HR", "Building A"), ("IT", "Building B"), ("Finance", "Building C")],
    ["dept", "location"]
)

# Inner join
df.join(dept_info, on="dept", how="inner").show()

# Left join
df.join(dept_info, on="dept", how="left").show()

# Left semi join — only left columns, but only rows with a match
df.join(dept_info, on="dept", how="left_semi").show()

# Left anti join — left rows with NO matching dept in right
df.join(dept_info, on="dept", how="left_anti").show()

# Cross join — cartesian product
df.select("name").crossJoin(dept_info.select("location")).show()

+----+---+-----+------+----------+-----------+----------+
|dept| id| name|salary| join_date|      email|  location|
+----+---+-----+------+----------+-----------+----------+
|  HR|  4| Dave|  null|2023-04-05|       null|Building A|
|  HR|  1|Alice|  5000|2023-01-15|       null|Building A|
|  IT|  3|Carol|  7000|2023-03-10|carol@x.com|Building B|
|  IT|  5|  Eve|  6000|2023-05-01|  eve@x.com|Building B|
|  IT|  2|  Bob|  6000|2023-02-20|  bob@x.com|Building B|
+----+---+-----+------+----------+-----------+----------+

+----+---+-----+------+----------+-----------+----------+
|dept| id| name|salary| join_date|      email|  location|
+----+---+-----+------+----------+-----------+----------+
|  HR|  1|Alice|  5000|2023-01-15|       null|Building A|
|  IT|  2|  Bob|  6000|2023-02-20|  bob@x.com|Building B|
|  HR|  4| Dave|  null|2023-04-05|       null|Building A|
|  IT|  3|Carol|  7000|2023-03-10|carol@x.com|Building B|
|  IT|  5|  Eve|  6000|2023-05-01|  eve@x.com|Building B|
+----+---+---

In [4]:
w = Window.partitionBy("dept").orderBy(F.desc("salary"))

df.withColumn("row_number", F.row_number().over(w)) \
  .withColumn("rank", F.rank().over(w)) \
  .withColumn("dense_rank", F.dense_rank().over(w)) \
  .withColumn("lag_salary", F.lag("salary", 1).over(w)) \
  .withColumn("lead_salary", F.lead("salary", 1).over(w)) \
  .show()

# Running total (cumulative sum) within each department, ordered by join_date
w_cum = Window.partitionBy("dept").orderBy("join_date") \
              .rowsBetween(Window.unboundedPreceding, Window.currentRow)

df.withColumn("running_total", F.sum("salary").over(w_cum)).show()

+---+-----+----+------+----------+-----------+----------+----+----------+----------+-----------+
| id| name|dept|salary| join_date|      email|row_number|rank|dense_rank|lag_salary|lead_salary|
+---+-----+----+------+----------+-----------+----------+----+----------+----------+-----------+
|  1|Alice|  HR|  5000|2023-01-15|       null|         1|   1|         1|      null|       null|
|  4| Dave|  HR|  null|2023-04-05|       null|         2|   2|         2|      5000|       null|
|  3|Carol|  IT|  7000|2023-03-10|carol@x.com|         1|   1|         1|      null|       6000|
|  2|  Bob|  IT|  6000|2023-02-20|  bob@x.com|         2|   2|         2|      7000|       6000|
|  5|  Eve|  IT|  6000|2023-05-01|  eve@x.com|         3|   2|         2|      6000|       null|
+---+-----+----+------+----------+-----------+----------+----+----------+----------+-----------+

+---+-----+----+------+----------+-----------+-------------+
| id| name|dept|salary| join_date|      email|running_total|
+---

In [5]:
# Find nulls
df.filter(F.col("salary").isNull()).show()
df.filter(F.col("salary").isNotNull()).show()

# Fill nulls — single value for all columns
df.fillna(0).show()

# Fill nulls per-column
df.fillna({"salary": 0, "email": "unknown@x.com"}).show()

# Drop rows with any null
df.dropna().show()

# Drop rows only if ALL columns are null, or only check specific columns
df.dropna(how="all").show()
df.dropna(subset=["salary"]).show()

# Coalesce — pick first non-null across columns
df.withColumn("contact", F.coalesce("email", F.lit("no_email"))).show()

+---+----+----+------+----------+-----+
| id|name|dept|salary| join_date|email|
+---+----+----+------+----------+-----+
|  4|Dave|  HR|  null|2023-04-05| null|
+---+----+----+------+----------+-----+

+---+-----+----+------+----------+-----------+
| id| name|dept|salary| join_date|      email|
+---+-----+----+------+----------+-----------+
|  1|Alice|  HR|  5000|2023-01-15|       null|
|  2|  Bob|  IT|  6000|2023-02-20|  bob@x.com|
|  3|Carol|  IT|  7000|2023-03-10|carol@x.com|
|  5|  Eve|  IT|  6000|2023-05-01|  eve@x.com|
+---+-----+----+------+----------+-----------+

+---+-----+----+------+----------+-----------+
| id| name|dept|salary| join_date|      email|
+---+-----+----+------+----------+-----------+
|  1|Alice|  HR|  5000|2023-01-15|       null|
|  2|  Bob|  IT|  6000|2023-02-20|  bob@x.com|
|  3|Carol|  IT|  7000|2023-03-10|carol@x.com|
|  4| Dave|  HR|     0|2023-04-05|       null|
|  5|  Eve|  IT|  6000|2023-05-01|  eve@x.com|
+---+-----+----+------+----------+-----------+

In [6]:
df2 = df.withColumn("join_date", F.to_date("join_date", "yyyy-MM-dd"))

df2.withColumn("today", F.current_date()) \
   .withColumn("days_since_joining", F.datediff(F.current_date(), F.col("join_date"))) \
   .withColumn("unix_ts", F.unix_timestamp("join_date")) \
   .withColumn("year", F.year("join_date")) \
   .withColumn("month", F.month("join_date")) \
   .withColumn("plus_30_days", F.date_add("join_date", 30)) \
   .show()

# Convert epoch seconds back to a readable timestamp
df2.withColumn("unix_ts", F.unix_timestamp("join_date")) \
   .withColumn("back_to_date", F.from_unixtime("unix_ts", "yyyy-MM-dd")) \
   .show()

+---+-----+----+------+----------+-----------+----------+------------------+----------+----+-----+------------+
| id| name|dept|salary| join_date|      email|     today|days_since_joining|   unix_ts|year|month|plus_30_days|
+---+-----+----+------+----------+-----------+----------+------------------+----------+----+-----+------------+
|  1|Alice|  HR|  5000|2023-01-15|       null|2026-09-05|              1329|1673740800|2023|    1|  2023-02-14|
|  2|  Bob|  IT|  6000|2023-02-20|  bob@x.com|2026-09-05|              1293|1676851200|2023|    2|  2023-03-22|
|  3|Carol|  IT|  7000|2023-03-10|carol@x.com|2026-09-05|              1275|1678406400|2023|    3|  2023-04-09|
|  4| Dave|  HR|  null|2023-04-05|       null|2026-09-05|              1249|1680652800|2023|    4|  2023-05-05|
|  5|  Eve|  IT|  6000|2023-05-01|  eve@x.com|2026-09-05|              1223|1682899200|2023|    5|  2023-05-31|
+---+-----+----+------+----------+-----------+----------+------------------+----------+----+-----+------

In [7]:
df.withColumn("full_info", F.concat(F.col("name"), F.lit(" - "), F.col("dept"))) \
  .withColumn("full_info_ws", F.concat_ws(" | ", "name", "dept")) \
  .withColumn("email_parts", F.split(F.coalesce("email", F.lit("")), "@")) \
  .withColumn("masked_email", F.regexp_replace(F.coalesce("email", F.lit("")), r"@.*", "@***")) \
  .withColumn("name_trimmed", F.trim(F.col("name"))) \
  .show(truncate=False)

# Extract username part before @ using split
df.withColumn("username", F.split(F.coalesce("email", F.lit("na@na")), "@").getItem(0)).show()

+---+-----+----+------+----------+-----------+----------+------------+--------------+------------+------------+
|id |name |dept|salary|join_date |email      |full_info |full_info_ws|email_parts   |masked_email|name_trimmed|
+---+-----+----+------+----------+-----------+----------+------------+--------------+------------+------------+
|1  |Alice|HR  |5000  |2023-01-15|null       |Alice - HR|Alice | HR  |[]            |            |Alice       |
|2  |Bob  |IT  |6000  |2023-02-20|bob@x.com  |Bob - IT  |Bob | IT    |[bob, x.com]  |bob@***     |Bob         |
|3  |Carol|IT  |7000  |2023-03-10|carol@x.com|Carol - IT|Carol | IT  |[carol, x.com]|carol@***   |Carol       |
|4  |Dave |HR  |null  |2023-04-05|null       |Dave - HR |Dave | HR   |[]            |            |Dave        |
|5  |Eve  |IT  |6000  |2023-05-01|eve@x.com  |Eve - IT  |Eve | IT    |[eve, x.com]  |eve@***     |Eve         |
+---+-----+----+------+----------+-----------+----------+------------+--------------+------------+------

In [8]:
# Simple distinct (all columns)
df.select("dept").distinct().show()

# dropDuplicates on subset of columns
df.dropDuplicates(["dept"]).show()

# Reliable dedup: keep the highest-salary row per department
w = Window.partitionBy("dept").orderBy(F.desc("salary"))
df.withColumn("rn", F.row_number().over(w)) \
  .filter(F.col("rn") == 1) \
  .drop("rn") \
  .show()  

+----+
|dept|
+----+
|  HR|
|  IT|
+----+

+---+-----+----+------+----------+-----------+
| id| name|dept|salary| join_date|      email|
+---+-----+----+------+----------+-----------+
|  4| Dave|  HR|  null|2023-04-05|       null|
|  3|Carol|  IT|  7000|2023-03-10|carol@x.com|
+---+-----+----+------+----------+-----------+

+---+-----+----+------+----------+-----------+
| id| name|dept|salary| join_date|      email|
+---+-----+----+------+----------+-----------+
|  1|Alice|  HR|  5000|2023-01-15|       null|
|  3|Carol|  IT|  7000|2023-03-10|carol@x.com|
+---+-----+----+------+----------+-----------+



In [9]:
df.createOrReplaceTempView("employees")

result = spark.sql("""
    SELECT dept,
           COUNT(*) AS emp_count,
           AVG(salary) AS avg_salary
    FROM employees
    WHERE salary IS NOT NULL
    GROUP BY dept
    ORDER BY avg_salary DESC
""")
result.show()

# You can freely mix SQL and DataFrame API — result of spark.sql() is a normal DataFrame
result.filter(F.col("emp_count") > 1).show()

# Window function in SQL too
spark.sql("""
    SELECT name, dept, salary,
           RANK() OVER (PARTITION BY dept ORDER BY salary DESC) AS rnk
    FROM employees
""").show()

+----+---------+-----------------+
|dept|emp_count|       avg_salary|
+----+---------+-----------------+
|  IT|        3|6333.333333333333|
|  HR|        1|           5000.0|
+----+---------+-----------------+

+----+---------+-----------------+
|dept|emp_count|       avg_salary|
+----+---------+-----------------+
|  IT|        3|6333.333333333333|
+----+---------+-----------------+

+-----+----+------+---+
| name|dept|salary|rnk|
+-----+----+------+---+
|Alice|  HR|  5000|  1|
| Dave|  HR|  null|  2|
|Carol|  IT|  7000|  1|
|  Bob|  IT|  6000|  2|
|  Eve|  IT|  6000|  2|
+-----+----+------+---+

